# NB15 — Panel-Bazli Egitim & Stacking Deneyi

**TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

Yarisma artik her panel (CFTR / KANSER / PAH) icin **ayri model** istiyor. Bu defter
dort egitim senaryosunu, dort model tipi uzerinde (NN, DNN, LightGBM, CatBoost)
panel-bazli olarak degerlendirir. **Birincil metrik: pathogenic (Label=1) F1.**

## Senaryolar

| # | Egitim verisi | Test verisi | Modeller |
|---|---|---|---|
| **S1** | Panelin %50'si (patho %50 + benign %50) **+ dengelenmis MASTER cekirdegi (625 patho / 625 benign)** | Panelin kalan %50'si | LightGBM, CatBoost, NN, DNN |
| **S2** | Panelin %50'si **+ ayni boyutta MASTER** (panel-train'i tam 2x'e cikarir, panelin patho/benign oranini korur) | Panelin kalan %50'si | LightGBM, CatBoost, NN, DNN |
| **S3** | NN, dengelenmis MASTER cekirdegi ile **pretrain** -> her panelin %50'si ile **finetune** | Panelin kalan %50'si | NN (ShallowMLP) |
| **S4** | S3'un ayni — fakat **DNN** (DeepMLP) | Panelin kalan %50'si | DNN (DeepMLP) |

**Onemli kurallar:**
- Test setlerinde **MASTER YOK** — yalnizca o panelin kendi test yarisi.
- Panel %50/%50 split **stratified** (patho ve benign ayri ayri yariya bolunur).
- Tum paneller ayni sabit **625/625 MASTER cekirdegini** paylasir (SEED=42 ile bir kez undersample edilir).
- Imputation / encoder / scaler **yalniz train uzerinde fit**, test'e transform (sizinti yok).
- **Hiperparametre optimizasyonu**: dort model tipi de **3-fold StratifiedKFold grid search** yapar
  (LightGBM=12, CatBoost=12, NN=4, DNN=4 kombo). NN/DNN combo'su S1/S2'de train uzerinde,
  S3/S4'te panel finetune verisi uzerinde CV-F1 ile secilir. Tiny panellerde fold sayisi
  guvenli sekilde dusurulur.
- Threshold, train icindeki dahili validasyon ile F1-max secilir; **test'e dokunmaz**.

## Missing-flag stratejisi (NB14'ten devralinan en faydali sonuc)

NB14 raporu uc missing senaryosunu karsilastirdi (panel-transfer ortalama F1):

| Senaryo | Aciklama | Panel mean F1 |
|---|---|---|
| M1 | Flag yok, medyan impute | 0.9165 |
| **M3** | **`is_missing_*` flag + medyan impute (orijinal sutun korunur)** | **0.9210 (en iyi)** |
| M5 | Flag + >%50 NaN drop / <=%50 medyan | 0.9188 |

**M3** hem panel-transferde en yuksek ortalama F1'i verdi hem de MASTER hold-out'ta
en iyiyle baglanti halinde. Bu defterde **M3 stratejisi** kullanilir:
yuksek-NaN sutunlar icin `is_missing_*` bayragi turetilir VE tum sayisal sutunlar
train-medyani ile doldurulur.

## Veri butunlugu notu (Variant_ID)

EDA: Ayni `Variant_ID` MASTER ile alt-panellerde gorunse de bunlar **farkli varyantlardir**
(ortalama 41-150 ozellik hucresi farkli). Yani Variant_ID global primary key DEGIL.
Yalniz cok az sayida satir (KANSER'de 3, PAH'ta 3, CFTR'de 0) MASTER ile **birebir ayni**
(tum ozellikler + label) — bunlar gercek kopya olup panelden drop edilir.


## v2 GUNCELLEMESI (final test dagilimi + NN iyilestirme)

**KRITIK:** Yarisma final test seti ~**%80 benign / %20 patho** olacak — train'in TERSI.
Metrik yine pathogenic F1. Bu surumde:
- **Threshold UC modda** secilir: `f1_raw` (eski, train dengesi), `f1_8020`, `mcc_8020`
  (final %80/20 dagilima yeniden orneklenmis train havuzunda; benign-aware).
- **Test iki dagilimda** raporlanir: %50/50 (mevcut panel) + **bootstrap %80/20**
  (benign sabit, patho downsample, N tekrar, %95 CI). Birincil = %80/20 F1.
- **NN/DNN iyilestirme**: CV 3->5 + FocalLoss + class weighting + early stopping +
  kucuk/duzenli mimari (SmallMLP). v1'de NN/DNN overfit ediyordu.


In [1]:
# Cell 1: Imports & Config
import os
import sys
import warnings
import json
from copy import deepcopy
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

warnings.filterwarnings("ignore")

# Proje kokunu path'e ekle (notebook farkli dizinden calistirilabilir)
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR, PROJECT_ROOT as CFG_ROOT
from src import columns_real as CR
from src.models import (
    MLP3Layer, DeepMLP,
    NN_FIXED_FAST, NN_GRID_FAST,
    DNN_FIXED_FAST, DNN_GRID_FAST,
    LGBM_GRID, LGBM_FIXED,
    CB_GRID, CB_FIXED,
)
from src.metrics import optimize_threshold

import lightgbm as lgb
from catboost import CatBoostClassifier

# Tekrarlanabilirlik
np.random.seed(SEED)
torch.manual_seed(SEED)

# Cihaz (macOS'ta CPU'ya duser)
try:
    import torch_directml
    DEVICE = torch_directml.device()
except Exception:
    DEVICE = torch.device("cpu")

# --- Deney sabitleri ---
PANELS = ["CFTR", "KANSER", "PAH"]
MASTER_CORE_POS = 625          # S1: dengelenmis MASTER cekirdegi pathogenic sayisi
MASTER_CORE_NEG = 625          # S1: dengelenmis MASTER cekirdegi benign sayisi
HIGH_MISSING_THRESHOLD = 0.50  # M3: bu oranin uzerinde NaN olan sutun -> is_missing flag
PANEL_SPLIT_FRAC = 0.50        # panelin yarisi train, yarisi test

# --- Cikti yollari ---
DATA_DIR = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v5_panel_specific")
MODELS_DIR = os.path.join(PROJECT_ROOT, "models", "v5_panel_specific")
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DEVICE       : {DEVICE}")
print(f"SEED         : {SEED}")
print(f"Paneller     : {PANELS}")
print(f"MASTER core  : {MASTER_CORE_POS} patho / {MASTER_CORE_NEG} benign")
print(f"Missing thr  : {HIGH_MISSING_THRESHOLD} (M3 stratejisi)")


PROJECT_ROOT : c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
DEVICE       : privateuseone:0
SEED         : 42
Paneller     : ['CFTR', 'KANSER', 'PAH']
MASTER core  : 625 patho / 625 benign
Missing thr  : 0.5 (M3 stratejisi)


In [2]:
# Cell 2: Veri Yukleme & Variant_ID Butunluk Kontrolu
ID_COL = CR.ID_COL          # 'Variant_ID'
TARGET = CR.TARGET_COL      # 'Label'

def load_panel(name):
    return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))

master = load_panel("MASTER")
panels_raw = {p: load_panel(p) for p in PANELS}

feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

print("=== Ham veri ===")
print(f"MASTER: {master.shape}  Label={master[TARGET].value_counts().to_dict()}")
for p, df in panels_raw.items():
    print(f"{p}: {df.shape}  Label={df[TARGET].value_counts().to_dict()}")

# --- Cross-panel BIREBIR AYNI satir tespiti ---
# Ayni Variant_ID cogunlukla FARKLI varyanttir; yalnizca tum ozellik+label
# birebir ayni olanlar gercek kopyadir ve panelden drop edilir.
master_index = {}
for _, row in master.iterrows():
    key = (row[ID_COL],) + tuple(
        (np.nan if pd.isna(v) else v) for v in row[feature_cols_all].values
    )
    master_index[key] = row[TARGET]

panels = {}
dup_report = {}
for p, df in panels_raw.items():
    drop_idx = []
    for idx, row in df.iterrows():
        key = (row[ID_COL],) + tuple(
            (np.nan if pd.isna(v) else v) for v in row[feature_cols_all].values
        )
        if key in master_index and master_index[key] == row[TARGET]:
            drop_idx.append(idx)
    cleaned = df.drop(index=drop_idx).reset_index(drop=True)
    panels[p] = cleaned
    dup_report[p] = len(drop_idx)
    print(f"{p}: MASTER ile birebir ayni {len(drop_idx)} satir drop edildi -> "
          f"{cleaned.shape[0]} satir kaldi  Label={cleaned[TARGET].value_counts().to_dict()}")

print("\\nNot: Ayni Variant_ID'ye sahip ama ozellikleri FARKLI satirlar KORUNDU "
      "(bunlar farkli varyantlar).")


=== Ham veri ===
MASTER: (2931, 353)  Label={1: 2149, 0: 782}
CFTR: (111, 353)  Label={1: 90, 0: 21}
KANSER: (388, 353)  Label={1: 268, 0: 120}
PAH: (372, 353)  Label={1: 310, 0: 62}
CFTR: MASTER ile birebir ayni 0 satir drop edildi -> 111 satir kaldi  Label={1: 90, 0: 21}
KANSER: MASTER ile birebir ayni 3 satir drop edildi -> 385 satir kaldi  Label={1: 265, 0: 120}
PAH: MASTER ile birebir ayni 3 satir drop edildi -> 369 satir kaldi  Label={1: 307, 0: 62}
\nNot: Ayni Variant_ID'ye sahip ama ozellikleri FARKLI satirlar KORUNDU (bunlar farkli varyantlar).


In [3]:
# Cell 3: Sutun Temizligi (sabit + birebir ayni sutun ciftleri)
# CLAUDE.md: constant ve duplicate sutunlar TRAIN'den sonra tespit edilip drop.
# Burada tum egitimlerin ortak girdisi olacak global feature listesini
# MASTER uzerinden belirliyoruz (panel test setleri ayni semayi paylasmali).
# Feature engineering YOK.

# 1) Sabit sutunlar (MASTER'da nunique<=1)
constant_cols = CR.get_constant_cols(master[feature_cols_all])

# 2) Birebir ayni sutun ciftleri (CAT_3==CAT_5 dahil) -> her ciftin 2.'si drop
dup_pairs = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop = sorted({b for (a, b) in dup_pairs})

drop_cols = sorted(set(constant_cols) | set(dup_drop))
feature_cols = [c for c in feature_cols_all if c not in drop_cols]

print(f"Sabit sutunlar ({len(constant_cols)}): {constant_cols}")
print(f"Birebir ayni ciftler: {dup_pairs}")
print(f"   -> drop edilen ikinciler: {dup_drop}")
print(f"\\nToplam drop: {len(drop_cols)} sutun")
print(f"Kalan feature: {len(feature_cols)} (ID/Label haric, FE yok)")

# Kategorik sutunlari survive eden listeden tespit et
CAT_LIKE = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in feature_cols]
NUM_COLS = [c for c in feature_cols if c not in CAT_LIKE]
print(f"Kategorik (CAT+AA): {len(CAT_LIKE)} -> {CAT_LIKE}")
print(f"Sayisal           : {len(NUM_COLS)}")


Sabit sutunlar (57): ['AL_80', 'AL_101', 'AL_104', 'AL_107', 'AL_110', 'AL_113', 'AL_116', 'AL_119', 'AL_122', 'AL_125', 'AL_128', 'AL_131', 'AL_134', 'AL_137', 'AL_140', 'AL_143', 'AL_146', 'AL_149', 'AL_152', 'AL_155', 'AL_158', 'AL_161', 'AL_164', 'AL_167', 'AL_170', 'AL_173', 'AL_176', 'AL_179', 'AL_182', 'AL_185', 'AL_191', 'AL_195', 'AL_197', 'AL_208', 'AL_212', 'AL_220', 'AL_231', 'AL_233', 'AL_236', 'AL_244', 'AL_248', 'AL_256', 'AL_263', 'AL_272', 'AL_276', 'AL_280', 'AL_284', 'AL_292', 'AL_299', 'AL_303', 'AL_307', 'AL_309', 'AL_312', 'AL_316', 'AL_320', 'AL_324', 'AL_332']
Birebir ayni ciftler: [('CAT_3', 'CAT_5'), ('AL_101', 'AL_104'), ('AL_101', 'AL_107'), ('AL_101', 'AL_110'), ('AL_101', 'AL_113'), ('AL_101', 'AL_116'), ('AL_101', 'AL_119'), ('AL_101', 'AL_122'), ('AL_101', 'AL_125'), ('AL_101', 'AL_128'), ('AL_101', 'AL_131'), ('AL_101', 'AL_134'), ('AL_101', 'AL_137'), ('AL_101', 'AL_140'), ('AL_101', 'AL_143'), ('AL_101', 'AL_146'), ('AL_101', 'AL_149'), ('AL_101', 'AL

In [4]:
# Cell 4: M3 Preprocessing (fit-on-train, transform) — NB14'ten devralinan strateji
# M3 = is_missing_* bayragi (yuksek-NaN sutunlar) + tum sayisal sutunlar medyan impute.
# Kategorik NaN -> 'MISSING' / AA icin 'X' token.
# Bir "preprocessor" sozlugu fit edilir (median, le-maps, high-miss cols, flag-source),
# sonra hem train hem test ayni sozlukle transform edilir.

AA_UNK = CR.AA_UNKNOWN_TOKEN  # 'X'

def fit_preprocessor(X_train):
    # X_train (feature_cols) uzerinde M3 onisleyiciyi fit eder.
    pp = {}
    # Yuksek-NaN sutunlar (train'de) -> is_missing flag uretilecek kaynaklar
    miss_ratio = X_train[feature_cols].isna().mean()
    pp["flag_source"] = miss_ratio[miss_ratio > HIGH_MISSING_THRESHOLD].index.tolist()
    # Sayisal medyanlar (train)
    pp["median"] = {c: X_train[c].median() for c in NUM_COLS}
    return pp

def transform_X(X, pp, encode="tree"):
    # M3 transform.
    # encode='tree' -> LightGBM/CatBoost icin: kategorikler string ('MISSING' fill),
    #                  sayisallar medyan-impute + is_missing flag.
    # encode='le'   -> sayisal matris (kategorikler LabelEncode, NN/DNN icin).
    # Donus: (DataFrame, cat_cols)
    X = X.copy()
    out = pd.DataFrame(index=X.index)

    # is_missing bayraklari (flag_source sutunlarindan)
    flag_df = {}
    for c in pp["flag_source"]:
        flag_df[CR.get_missing_mask_col_name(c)] = X[c].isna().astype(int).values

    # Sayisal sutunlar: medyan impute
    for c in NUM_COLS:
        out[c] = pd.to_numeric(X[c], errors="coerce").fillna(pp["median"][c]).astype(float).values

    # Kategorik sutunlar
    for c in CAT_LIKE:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = X[c].astype("object").where(~X[c].isna(), fill).astype(str).values

    for fc, vals in flag_df.items():
        out[fc] = vals

    cat_cols = list(CAT_LIKE)
    return out, cat_cols

# Hizli dogrulama: tum MASTER uzerinde fit/transform
_pp = fit_preprocessor(master)
_Xt, _cc = transform_X(master[feature_cols], _pp, encode="tree")
print(f"flag_source ({len(_pp['flag_source'])} sutun >%{int(HIGH_MISSING_THRESHOLD*100)} NaN)")
print(f"Transform sonrasi: {_Xt.shape[1]} sutun "
      f"(= {len(NUM_COLS)} sayisal + {len(CAT_LIKE)} kategorik + "
      f"{len(_pp['flag_source'])} is_missing flag)")
print(f"NaN kaldi mi? {_Xt.isna().any().any()}")


flag_source (140 sutun >%50 NaN)
Transform sonrasi: 428 sutun (= 281 sayisal + 7 kategorik + 140 is_missing flag)
NaN kaldi mi? False


In [5]:
# Cell 5: Split & Dataset Olusturucular
# - panel_5050_split: panelin patho/benign'ini AYRI AYRI yariya boler (stratified)
# - MASTER 625/625 cekirdek: bir kez ortak undersample (tum paneller paylasir)
# - build_s1 / build_s2: senaryolara gore train cercevesi olusturur (test = panel test yarisi)

rng = np.random.RandomState(SEED)

def panel_5050_split(df):
    # Panelin patho ve benign'ini ayri ayri %50 train / %50 test'e ayirir.
    pos = df[df[TARGET] == 1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET] == 0].sample(frac=1.0, random_state=SEED)
    n_pos_tr = int(round(len(pos) * PANEL_SPLIT_FRAC))
    n_neg_tr = int(round(len(neg) * PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:n_pos_tr], neg.iloc[:n_neg_tr]]).sample(frac=1.0, random_state=SEED)
    te = pd.concat([pos.iloc[n_pos_tr:], neg.iloc[n_neg_tr:]]).sample(frac=1.0, random_state=SEED)
    return tr.reset_index(drop=True), te.reset_index(drop=True)

# Sabit, dengelenmis MASTER cekirdegi (tum paneller ayni cekirdegi kullanir)
def make_master_core(n_pos, n_neg):
    pos = master[master[TARGET] == 1]
    neg = master[master[TARGET] == 0]
    n_pos = min(n_pos, len(pos))
    n_neg = min(n_neg, len(neg))
    core = pd.concat([
        pos.sample(n=n_pos, random_state=SEED),
        neg.sample(n=n_neg, random_state=SEED),
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    return core

MASTER_CORE = make_master_core(MASTER_CORE_POS, MASTER_CORE_NEG)  # S1 cekirdegi
print(f"MASTER cekirdek (S1): {MASTER_CORE.shape[0]} satir  "
      f"Label={MASTER_CORE[TARGET].value_counts().to_dict()}")

# Panel split'lerini bir kez hesapla ve sakla (tum senaryolar ayni split'i kullanir)
PANEL_SPLITS = {}
for p in PANELS:
    tr, te = panel_5050_split(panels[p])
    PANEL_SPLITS[p] = (tr, te)
    print(f"{p}: train={tr.shape[0]} "
          f"(patho={int((tr[TARGET]==1).sum())},benign={int((tr[TARGET]==0).sum())})  "
          f"test={te.shape[0]} "
          f"(patho={int((te[TARGET]==1).sum())},benign={int((te[TARGET]==0).sum())})")

def build_s1(panel_train):
    # S1: panel train + sabit 625/625 MASTER cekirdegi.
    return pd.concat([panel_train, MASTER_CORE], ignore_index=True)

def build_s2(panel_train):
    # S2: panel train + ayni boyutta MASTER (panel-train'i 2x yapar).
    # MASTER eklemesi panelin kendi patho/benign sayisini AYNALAYARAK yapilir
    # (yani toplam train tam olarak 2x panel-train, sinif orani korunur).
    n_pos = int((panel_train[TARGET] == 1).sum())
    n_neg = int((panel_train[TARGET] == 0).sum())
    mpos = master[master[TARGET] == 1].sample(n=min(n_pos, (master[TARGET]==1).sum()), random_state=SEED)
    mneg = master[master[TARGET] == 0].sample(n=min(n_neg, (master[TARGET]==0).sum()), random_state=SEED)
    return pd.concat([panel_train, mpos, mneg], ignore_index=True)


MASTER cekirdek (S1): 1250 satir  Label={0: 625, 1: 625}
CFTR: train=55 (patho=45,benign=10)  test=56 (patho=45,benign=11)
KANSER: train=192 (patho=132,benign=60)  test=193 (patho=133,benign=60)
PAH: train=185 (patho=154,benign=31)  test=184 (patho=153,benign=31)


In [6]:
# Cell 5b: Degerlendirme Altyapisi (benign-aware threshold + %80/20 bootstrap)
# ANA AMAC: Final test seti ~%80 benign / ~%20 patho olacak (train'in TERSI).
# Bu yuzden iki sey ekliyoruz:
#   (1) Threshold'u SADECE pathogenic-F1-max yerine, final dagilima (%80/20)
#       yeniden orneklenmis train havuzunda hem F1-max hem MCC-max ile sec.
#   (2) Test degerlendirmesini hem %50/50 (mevcut panel test) hem de bootstrap
#       %80/20 (benign sabit, patho downsample, N tekrar) olarak raporla.
# Modeller artik ham olasilik (ptr/pte) donduruyor; threshold/metrik burada.

from sklearn.metrics import matthews_corrcoef

FINAL_BENIGN_FRAC = 0.80     # final test benign orani
N_BOOT = 50                  # %80/20 bootstrap tekrar sayisi
BOOT_SEED = SEED

def _f1_pos(y_true, y_pred):
    return f1_score(y_true, y_pred, pos_label=1, zero_division=0)

def _metrics_at(y_true, y_prob, thr):
    yp = (y_prob >= thr).astype(int)
    return {
        "f1": _f1_pos(y_true, yp),
        "precision": precision_score(y_true, yp, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, yp, pos_label=1, zero_division=0),
        "mcc": matthews_corrcoef(y_true, yp) if len(np.unique(y_true)) > 1 else 0.0,
    }

def _resample_8020(y, prob, rng):
    # Tum benign'leri tut + n_benign/4 kadar patho (yerine koymali) -> ~%80/20.
    y = np.asarray(y); prob = np.asarray(prob)
    neg_idx = np.where(y == 0)[0]
    pos_idx = np.where(y == 1)[0]
    if len(neg_idx) == 0 or len(pos_idx) == 0:
        return y, prob
    n_pos = max(1, int(round(len(neg_idx) * (1 - FINAL_BENIGN_FRAC) / FINAL_BENIGN_FRAC)))
    pos_samp = rng.choice(pos_idx, size=n_pos, replace=True)
    keep = np.concatenate([neg_idx, pos_samp])
    return y[keep], prob[keep]

def select_threshold(y_train, p_train, mode="f1_8020"):
    # Threshold'u TRAIN olasiliklarindan secer (test'e dokunmaz).
    #   mode='f1_8020'  -> train'i %80/20'ye yeniden ornekle, F1-max esik
    #   mode='mcc_8020' -> ayni havuzda MCC-max esik (benign'e daha cok onem)
    #   mode='f1_raw'   -> mevcut optimize_threshold (geriye uyum / kiyas)
    y_train = np.asarray(y_train); p_train = np.asarray(p_train)
    if mode == "f1_raw":
        thr, _ = optimize_threshold(y_train, p_train)
        return float(thr)
    rng = np.random.RandomState(BOOT_SEED)
    yb, pb = _resample_8020(y_train, p_train, rng)
    best_thr, best_score = 0.5, -2.0
    for thr in np.arange(0.05, 0.95, 0.01):
        yp = (pb >= thr).astype(int)
        if mode == "f1_8020":
            score = _f1_pos(yb, yp)
        else:  # mcc_8020
            score = matthews_corrcoef(yb, yp) if len(np.unique(yb)) > 1 else 0.0
        if score > best_score:
            best_score, best_thr = score, thr
    return float(best_thr)

def bootstrap_8020_f1(y_test, p_test, thr, n_boot=N_BOOT):
    # Test setini N kez %80/20'ye yeniden ornekle, pathogenic-F1 dagilimi dondur.
    rng = np.random.RandomState(BOOT_SEED)
    f1s = []
    for _ in range(n_boot):
        yb, pb = _resample_8020(y_test, p_test, rng)
        f1s.append(_f1_pos(yb, (pb >= thr).astype(int)))
    f1s = np.array(f1s)
    return {"f1_8020_mean": float(f1s.mean()), "f1_8020_std": float(f1s.std()),
            "f1_8020_lo": float(np.percentile(f1s, 2.5)),
            "f1_8020_hi": float(np.percentile(f1s, 97.5))}

def evaluate_predictions(y_train, p_train, y_test, p_test, thr_mode="f1_8020"):
    # Merkezi degerlendirme: threshold sec (train'den), hem %50/50 hem %80/20 raporla.
    thr = select_threshold(y_train, p_train, mode=thr_mode)
    tr = _metrics_at(y_train, p_train, thr)        # train metrikleri (ayni esik)
    te = _metrics_at(y_test, p_test, thr)          # test %50/50 (mevcut panel dagilimi)
    boot = bootstrap_8020_f1(y_test, p_test, thr)  # test %80/20 bootstrap pathogenic-F1
    y_pred = (np.asarray(p_test) >= thr).astype(int)
    return {"thr": float(thr), "thr_mode": thr_mode,
            "train": tr, "test": te, "boot8020": boot,
            "y_true": np.asarray(y_test), "y_pred": y_pred}

print("Degerlendirme altyapisi tanimlandi: evaluate_predictions "
      f"(thr modlari: f1_8020/mcc_8020/f1_raw, bootstrap N={N_BOOT}, "
      f"final benign={int(FINAL_BENIGN_FRAC*100)}%)")


Degerlendirme altyapisi tanimlandi: evaluate_predictions (thr modlari: f1_8020/mcc_8020/f1_raw, bootstrap N=50, final benign=80%)


In [7]:
# Cell 6: Agac Model Egiticileri (LightGBM, CatBoost)
# Her egitici: train df + test df alir, M3 preprocessing'i train'de fit eder,
# 5-fold CV grid search ile en iyi combo'yu secer, FINAL modeli kurar ve
# train + test HAM OLASILIKLARINI dondurur. Threshold secimi + %50/50 ve %80/20
# degerlendirme MERKEZI evaluate_predictions'ta yapilir (Cell 5b).

from sklearn.model_selection import StratifiedKFold

N_CV = 5  # danisman onerisi: 3 -> 5 (combo secim kararliligi)

def _grid_combos(grid):
    import itertools
    keys = list(grid)
    for vals in itertools.product(*[grid[k] for k in keys]):
        yield dict(zip(keys, vals))

def _cv_splits(y, want=N_CV):
    # Tiny panellerde en kucuk sinifa gore guvenli fold sayisi.
    y = np.asarray(y)
    min_class = int(min((y == 0).sum(), (y == 1).sum()))
    return max(2, min(want, min_class)) if min_class >= 2 else 2

def _fit_lightgbm(train_df, test_df):
    # LightGBM'i CV grid search ile egitir; ham olasilik (ptr, pte) + importance doner.
    pp = fit_preprocessor(train_df)
    Xtr, cat_cols = transform_X(train_df[feature_cols], pp, encode="tree")
    Xte, _ = transform_X(test_df[feature_cols], pp, encode="tree")
    ytr = train_df[TARGET].reset_index(drop=True)
    Xtr = Xtr.reset_index(drop=True); Xte = Xte.reset_index(drop=True)
    for c in cat_cols:
        Xtr[c] = Xtr[c].astype("category")
        Xte[c] = pd.Categorical(Xte[c], categories=Xtr[c].cat.categories)

    best_f1, best_combo = -1, None
    ns = _cv_splits(ytr)
    for combo in _grid_combos(LGBM_GRID):
        params = {**LGBM_FIXED, **combo}
        skf = StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
        fold = []
        for tri, vai in skf.split(Xtr, ytr):
            m = lgb.LGBMClassifier(**params)
            m.fit(Xtr.iloc[tri], ytr.iloc[tri], categorical_feature=cat_cols)
            pv = m.predict_proba(Xtr.iloc[vai])[:, 1]
            _, f = optimize_threshold(ytr.iloc[vai], pv)
            fold.append(f)
        if np.mean(fold) > best_f1:
            best_f1, best_combo = np.mean(fold), combo

    model = lgb.LGBMClassifier(**{**LGBM_FIXED, **best_combo})
    model.fit(Xtr, ytr, categorical_feature=cat_cols)
    ptr = model.predict_proba(Xtr)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]
    importance = pd.Series(model.feature_importances_, index=Xtr.columns)
    return ytr.values, ptr, pte, importance

def _fit_catboost(train_df, test_df):
    pp = fit_preprocessor(train_df)
    Xtr, cat_cols = transform_X(train_df[feature_cols], pp, encode="tree")
    Xte, _ = transform_X(test_df[feature_cols], pp, encode="tree")
    ytr = train_df[TARGET].reset_index(drop=True)
    Xtr = Xtr.reset_index(drop=True); Xte = Xte.reset_index(drop=True)
    cat_idx = [list(Xtr.columns).index(c) for c in cat_cols]

    best_f1, best_combo = -1, None
    ns = _cv_splits(ytr)
    for combo in _grid_combos(CB_GRID):
        params = {**CB_FIXED, **combo}
        skf = StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
        fold = []
        for tri, vai in skf.split(Xtr, ytr):
            m = CatBoostClassifier(**params)
            m.fit(Xtr.iloc[tri], ytr.iloc[tri], cat_features=cat_idx, silent=True)
            pv = m.predict_proba(Xtr.iloc[vai])[:, 1]
            _, f = optimize_threshold(ytr.iloc[vai], pv)
            fold.append(f)
        if np.mean(fold) > best_f1:
            best_f1, best_combo = np.mean(fold), combo

    model = CatBoostClassifier(**{**CB_FIXED, **best_combo})
    model.fit(Xtr, ytr, cat_features=cat_idx, silent=True)
    ptr = model.predict_proba(Xtr)[:, 1]
    pte = model.predict_proba(Xte)[:, 1]
    importance = pd.Series(model.get_feature_importance(), index=Xtr.columns)
    return ytr.values, ptr, pte, importance

def train_lightgbm(train_df, test_df, thr_mode="f1_8020"):
    ytr, ptr, pte, importance = _fit_lightgbm(train_df, test_df)
    res = evaluate_predictions(ytr, ptr, test_df[TARGET].values, pte, thr_mode=thr_mode)
    res["importance"] = importance
    res["best_combo"] = ""
    return res

def train_catboost(train_df, test_df, thr_mode="f1_8020"):
    ytr, ptr, pte, importance = _fit_catboost(train_df, test_df)
    res = evaluate_predictions(ytr, ptr, test_df[TARGET].values, pte, thr_mode=thr_mode)
    res["importance"] = importance
    res["best_combo"] = ""
    return res

print(f"Agac egiticileri tanimlandi (CV={N_CV}, ham olasilik + merkezi eval): "
      "train_lightgbm, train_catboost")


Agac egiticileri tanimlandi (CV=5, ham olasilik + merkezi eval): train_lightgbm, train_catboost


In [8]:
# Cell 7: NN/DNN Yardimcilari (iyilestirilmis: CV5 + focal loss + early stopping + regularization)
# Literatur (Cell 0 / rapor): kucuk tabular veride NN overfit eder. Cozumler:
#   - CV 3 -> 5 (danisman onerisi; combo secim kararliligi)
#   - FocalLoss (src.focal_loss) + class weighting (pos_weight) -> azinlik benign'e odak
#     (final test %80 benign oldugu icin benign'i ogrenmek kritik)
#   - Validasyon-bazli EARLY STOPPING (sabit epoch yerine) -> overfit'i dogrudan kes
#   - Daha guclu regularization: yuksek dropout + weight_decay grid'de taranir
#   - Daha KUCUK mimari (NN_SMALL/DNN_SMALL) -> ~55-190 satirlik panele uygun
# Modeller artik HAM OLASILIK doner; threshold/metrik MERKEZI evaluate_predictions'ta.

import itertools as _itertools
from copy import deepcopy as _deepcopy
from src.focal_loss import FocalLoss

NN_CV = 5            # danisman onerisi
NN_MAX_EPOCHS = 80   # early stopping ile kesilecek ust sinir
NN_PATIENCE = 12     # validasyon F1 iyilesmezse dur
NN_VAL_FRAC = 0.25   # erken durdurma icin train-ici validasyon orani

# Kucuk, daha duzenli grid'ler (overfit'e karsi). Sabit buyuk h1/h2/h3 yerine
# kucuk hidden boyutlari + yuksek dropout + weight_decay taraniyor.
NN_GRID_REG = {
    "hidden": [64, 128],
    "dropout": [0.3, 0.5],
    "weight_decay": [1e-3],
    "lr": [1e-3],
    "focal_gamma": [2.0],
}
DNN_GRID_REG = {
    "hidden": [128],
    "n_layers": [2, 3],
    "dropout": [0.4, 0.5],
    "weight_decay": [1e-3],
    "lr": [1e-3],
    "focal_gamma": [2.0],
}

class SmallMLP(torch.nn.Module):
    # Kucuk, duzenli MLP. BatchNorm + Dropout 'variance shift' riskine karsi
    # BatchNorm yerine sadece Dropout + (opsiyonel) LayerNorm kullanir.
    def __init__(self, input_dim, hidden, n_layers, dropout):
        super().__init__()
        layers = []
        d = input_dim
        for _ in range(n_layers):
            layers += [torch.nn.Linear(d, hidden), torch.nn.ReLU(),
                       torch.nn.Dropout(dropout)]
            d = hidden
        layers += [torch.nn.Linear(d, 1)]
        self.net = torch.nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x).squeeze(-1)

def fit_nn_encoder(train_df):
    # NN icin: M3 transform + LabelEncode map + StandardScaler (train'de fit).
    pp = fit_preprocessor(train_df)
    Xtr_tree, cat_cols = transform_X(train_df[feature_cols], pp, encode="tree")
    le_maps = {}
    Xnum = Xtr_tree.copy()
    for c in cat_cols:
        le = LabelEncoder()
        Xnum[c] = le.fit_transform(Xnum[c].astype(str))
        le_maps[c] = {v: i for i, v in enumerate(le.classes_)}
    Xnum = Xnum.astype(np.float32)
    scaler = StandardScaler().fit(Xnum.values)
    return {"pp": pp, "cat_cols": cat_cols, "le_maps": le_maps,
            "scaler": scaler, "columns": list(Xnum.columns)}

def nn_matrix(df, enc):
    Xtree, _ = transform_X(df[feature_cols], enc["pp"], encode="tree")
    Xnum = Xtree.copy()
    for c in enc["cat_cols"]:
        Xnum[c] = Xnum[c].astype(str).map(enc["le_maps"][c]).fillna(-1)
    Xnum = Xnum[enc["columns"]].astype(np.float32)
    return torch.FloatTensor(enc["scaler"].transform(Xnum.values))

def _nn_grid_combos(kind):
    g = NN_GRID_REG if kind == "nn" else DNN_GRID_REG
    keys = list(g)
    for vals in _itertools.product(*[g[k] for k in keys]):
        yield dict(zip(keys, vals))

def _build_model(kind, input_dim, combo):
    if kind == "nn":
        return SmallMLP(input_dim, combo["hidden"], 2, combo["dropout"])
    return SmallMLP(input_dim, combo["hidden"], combo["n_layers"], combo["dropout"])

def _safe_n_splits(y, want=NN_CV):
    y = np.asarray(y)
    mc = int(min((y == 0).sum(), (y == 1).sum()))
    return max(2, min(want, mc)) if mc >= 2 else 1

def _train_es(model, Xtr, ytr, Xval, yval, combo, pos_weight):
    # FocalLoss + AdamW + validasyon-bazli early stopping. En iyi state'i geri yukler.
    crit = FocalLoss(alpha=0.25, gamma=combo["focal_gamma"], pos_weight=pos_weight)
    opt = torch.optim.AdamW(model.parameters(), lr=combo["lr"],
                            weight_decay=combo["weight_decay"])
    n = len(Xtr); bs = min(64, max(2, n - 1))
    best_f1, best_state, patience = -1.0, None, 0
    for _ in range(NN_MAX_EPOCHS):
        model.train()
        perm = torch.randperm(n)
        for i in range(0, n, bs):
            idx = perm[i:i + bs]
            if len(idx) < 2:
                continue
            opt.zero_grad()
            loss = crit(model(Xtr[idx]), ytr[idx])
            loss.backward(); opt.step()
        # validasyon F1 (thr=0.5, sadece early stopping icin)
        model.eval()
        with torch.no_grad():
            vp = torch.sigmoid(model(Xval)).numpy()
        vf1 = f1_score(yval.numpy(), (vp >= 0.5).astype(int), pos_label=1, zero_division=0)
        if vf1 > best_f1:
            best_f1, best_state, patience = vf1, _deepcopy(model.state_dict()), 0
        else:
            patience += 1
            if patience >= NN_PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model

def _proba(model, Xt):
    model.eval()
    with torch.no_grad():
        return torch.sigmoid(model(Xt)).numpy()

def _pos_weight(y):
    y = np.asarray(y)
    return torch.FloatTensor([(y == 0).sum() / max((y == 1).sum(), 1)])

def _nn_cv_select(kind, Xmat, y):
    # 5-fold (guvenli) CV grid search -> en iyi combo (val F1). Her fold'da early stopping.
    combos = list(_nn_grid_combos(kind))
    ns = _safe_n_splits(y)
    if ns < 2:
        return combos[0]
    yv = np.asarray(y)
    best_f1, best_combo = -1.0, combos[0]
    for combo in combos:
        skf = StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
        fold = []
        for tri, vai in skf.split(Xmat.numpy(), yv):
            m = _build_model(kind, Xmat.shape[1], combo)
            ytr_t = torch.FloatTensor(yv[tri].astype(np.float32))
            yval_t = torch.FloatTensor(yv[vai].astype(np.float32))
            m = _train_es(m, Xmat[tri], ytr_t, Xmat[vai], yval_t, combo, _pos_weight(yv[tri]))
            pv = _proba(m, Xmat[vai])
            fold.append(f1_score(yv[vai], (pv >= 0.5).astype(int), pos_label=1, zero_division=0))
        if np.mean(fold) > best_f1:
            best_f1, best_combo = np.mean(fold), combo
    return best_combo

def _final_fit(kind, Xtr_full, ytr_full, combo):
    # Final modeli: train'in bir parcasini early-stopping validasyonu olarak ayir.
    yv = np.asarray(ytr_full)
    from sklearn.model_selection import train_test_split
    strat = yv if min((yv == 0).sum(), (yv == 1).sum()) >= 2 else None
    tri, vai = train_test_split(np.arange(len(yv)), test_size=NN_VAL_FRAC,
                                random_state=SEED, stratify=strat)
    m = _build_model(kind, Xtr_full.shape[1], combo)
    m = _train_es(m, Xtr_full[tri], torch.FloatTensor(yv[tri].astype(np.float32)),
                  Xtr_full[vai], torch.FloatTensor(yv[vai].astype(np.float32)),
                  combo, _pos_weight(yv[tri]))
    return m

def train_nn_scratch(train_df, test_df, kind="nn", thr_mode="f1_8020"):
    # S1/S2: NN/DNN'i bastan train_df ile egit (CV grid search + early stopping).
    enc = fit_nn_encoder(train_df)
    Xtr = nn_matrix(train_df, enc); Xte = nn_matrix(test_df, enc)
    ytr_np = train_df[TARGET].values
    best_combo = _nn_cv_select(kind, Xtr, ytr_np)
    model = _final_fit(kind, Xtr, ytr_np, best_combo)
    ptr = _proba(model, Xtr); pte = _proba(model, Xte)
    res = evaluate_predictions(ytr_np, ptr, test_df[TARGET].values, pte, thr_mode=thr_mode)
    res["importance"] = None
    res["best_combo"] = str(best_combo)
    res["_ytr"], res["_ptr"] = ytr_np, ptr
    res["_yte"], res["_pte"] = test_df[TARGET].values, pte
    return res

def train_nn_finetune(pretrain_df, panel_train_df, test_df, kind="nn", thr_mode="f1_8020"):
    # S3/S4: pretrain (MASTER cekirdek) -> finetune (panel). CV combo panel'de secilir.
    fit_basis = pd.concat([pretrain_df, panel_train_df], ignore_index=True)
    enc = fit_nn_encoder(fit_basis)
    Xpre = nn_matrix(pretrain_df, enc); Xpan = nn_matrix(panel_train_df, enc)
    Xte = nn_matrix(test_df, enc)
    ypre = pretrain_df[TARGET].values; ypan = panel_train_df[TARGET].values

    best_combo = _nn_cv_select(kind, Xpan, ypan)

    # 1) Pretrain (MASTER): early stopping icin pretrain'in bir parcasi validasyon
    yv = np.asarray(ypre)
    from sklearn.model_selection import train_test_split
    tri, vai = train_test_split(np.arange(len(yv)), test_size=NN_VAL_FRAC,
                                random_state=SEED, stratify=yv)
    model = _build_model(kind, Xpre.shape[1], best_combo)
    model = _train_es(model, Xpre[tri], torch.FloatTensor(yv[tri].astype(np.float32)),
                      Xpre[vai], torch.FloatTensor(yv[vai].astype(np.float32)),
                      best_combo, _pos_weight(yv[tri]))
    # 2) Finetune (panel): dusuk lr, panel'i early-stop validasyonuyla
    ft_combo = dict(best_combo); ft_combo["lr"] = best_combo["lr"] * 0.1
    ypv = np.asarray(ypan)
    strat = ypv if min((ypv == 0).sum(), (ypv == 1).sum()) >= 2 else None
    ptri, pvai = train_test_split(np.arange(len(ypv)), test_size=NN_VAL_FRAC,
                                  random_state=SEED, stratify=strat)
    model = _train_es(model, Xpan[ptri], torch.FloatTensor(ypv[ptri].astype(np.float32)),
                      Xpan[pvai], torch.FloatTensor(ypv[pvai].astype(np.float32)),
                      ft_combo, _pos_weight(ypv[ptri]))

    ppan = _proba(model, Xpan); pte = _proba(model, Xte)
    res = evaluate_predictions(ypan, ppan, test_df[TARGET].values, pte, thr_mode=thr_mode)
    res["importance"] = None
    res["best_combo"] = str(best_combo)
    res["_ytr"], res["_ptr"] = np.asarray(ypan), ppan
    res["_yte"], res["_pte"] = test_df[TARGET].values, pte
    return res

print(f"NN/DNN yardimcilari (CV={NN_CV} + FocalLoss + early stopping + kucuk mimari) "
      "tanimlandi: train_nn_scratch, train_nn_finetune")
print(f"  NN grid kombo: {len(list(_nn_grid_combos('nn')))}  "
      f"DNN grid kombo: {len(list(_nn_grid_combos('dnn')))}")


NN/DNN yardimcilari (CV=5 + FocalLoss + early stopping + kucuk mimari) tanimlandi: train_nn_scratch, train_nn_finetune
  NN grid kombo: 4  DNN grid kombo: 4


In [9]:
# Cell 8: Ana Deney Dongusu (S1, S2, S3, S4)
# Her senaryo x her panel x ilgili model bir kez egitilir; ham olasiliklardan
# UC threshold modu da degerlendirilir (yeniden egitim YOK):
#   - f1_raw   : eski davranis (train dengesinde F1-max) -> kiyas/regresyon
#   - f1_8020  : threshold %80/20 yeniden-orneklenmis train'de F1-max
#   - mcc_8020 : ayni havuzda MCC-max (benign'e daha cok onem)
# Her mod icin %50/50 test + bootstrap %80/20 test pathogenic-F1 raporlanir.
# DETAIL[(scn,panel,model)] = f1_8020 modunun tam res'i (CM/importance icin).

THR_MODES = ["f1_raw", "f1_8020", "mcc_8020"]

rows = []
DETAIL = {}   # (scenario, panel, model) -> f1_8020 res (gorseller icin)

def _record_all_modes(scn, panel, model_name, ytr, ptr, yte, pte,
                      importance=None, best_combo=""):
    # Ham olasiliklardan 3 threshold modunu degerlendir, hepsini kaydet.
    for mode in THR_MODES:
        res = evaluate_predictions(ytr, ptr, yte, pte, thr_mode=mode)
        b = res["boot8020"]
        rows.append({
            "scenario": scn, "panel": panel, "model": model_name, "thr_mode": mode,
            "threshold": round(res["thr"], 3), "best_combo": str(best_combo),
            "train_f1": res["train"]["f1"], "train_precision": res["train"]["precision"],
            "train_recall": res["train"]["recall"], "train_mcc": res["train"]["mcc"],
            "test_f1_5050": res["test"]["f1"], "test_precision_5050": res["test"]["precision"],
            "test_recall_5050": res["test"]["recall"], "test_mcc_5050": res["test"]["mcc"],
            "test_f1_8020_mean": b["f1_8020_mean"], "test_f1_8020_std": b["f1_8020_std"],
            "test_f1_8020_lo": b["f1_8020_lo"], "test_f1_8020_hi": b["f1_8020_hi"],
            "n_test": len(yte),
        })
        if mode == "f1_8020":
            res["importance"] = importance
            res["best_combo"] = str(best_combo)
            DETAIL[(scn, panel, model_name)] = res
            print(f"  [{scn}|{panel}|{model_name:9s}] "
                  f"test F1(50/50)={res['test']['f1']:.3f}  "
                  f"F1(80/20)={b['f1_8020_mean']:.3f}+-{b['f1_8020_std']:.3f}  "
                  f"P={res['test']['precision']:.3f} R={res['test']['recall']:.3f} "
                  f"thr={res['thr']:.2f}")

# fit fonksiyonlari ham olasilik dondursun diye ince sarmalayicilar:
def _probs_lightgbm(train_df, test_df):
    ytr, ptr, pte, imp = _fit_lightgbm(train_df, test_df)
    return ytr, ptr, test_df[TARGET].values, pte, imp, ""
def _probs_catboost(train_df, test_df):
    ytr, ptr, pte, imp = _fit_catboost(train_df, test_df)
    return ytr, ptr, test_df[TARGET].values, pte, imp, ""

TREE_PROB_FNS = [("lightgbm", _probs_lightgbm), ("catboost", _probs_catboost)]

for panel in PANELS:
    p_tr, p_te = PANEL_SPLITS[panel]
    print(f"\\n========== Panel: {panel} ==========")

    # --- S1: panel + 625/625 MASTER cekirdek ---
    print("--- S1: panel + dengelenmis MASTER cekirdek (625/625) ---")
    s1 = build_s1(p_tr)
    for name, fn in TREE_PROB_FNS:
        ytr, ptr, yte, pte, imp, bc = fn(s1, p_te)
        _record_all_modes("S1", panel, name, ytr, ptr, yte, pte, imp, bc)
    for kind in ("nn", "dnn"):
        r = train_nn_scratch(s1, p_te, kind=kind, thr_mode="f1_8020")
        _record_all_modes("S1", panel, kind, r["_ytr"], r["_ptr"], r["_yte"], r["_pte"],
                          None, r["best_combo"])

    # --- S2: panel + ayni boyutta MASTER (2x panel) ---
    print("--- S2: panel + ayni boyutta MASTER (panel-train 2x) ---")
    s2 = build_s2(p_tr)
    for name, fn in TREE_PROB_FNS:
        ytr, ptr, yte, pte, imp, bc = fn(s2, p_te)
        _record_all_modes("S2", panel, name, ytr, ptr, yte, pte, imp, bc)
    for kind in ("nn", "dnn"):
        r = train_nn_scratch(s2, p_te, kind=kind, thr_mode="f1_8020")
        _record_all_modes("S2", panel, kind, r["_ytr"], r["_ptr"], r["_yte"], r["_pte"],
                          None, r["best_combo"])

    # --- S3: NN pretrain -> finetune ---
    print("--- S3: NN pretrain(MASTER core) -> finetune(panel) ---")
    r = train_nn_finetune(MASTER_CORE, p_tr, p_te, kind="nn", thr_mode="f1_8020")
    _record_all_modes("S3", panel, "nn", r["_ytr"], r["_ptr"], r["_yte"], r["_pte"],
                      None, r["best_combo"])

    # --- S4: DNN pretrain -> finetune ---
    print("--- S4: DNN pretrain(MASTER core) -> finetune(panel) ---")
    r = train_nn_finetune(MASTER_CORE, p_tr, p_te, kind="dnn", thr_mode="f1_8020")
    _record_all_modes("S4", panel, "dnn", r["_ytr"], r["_ptr"], r["_yte"], r["_pte"],
                      None, r["best_combo"])

results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(RESULTS_DIR, "panel_specific_results.csv"), index=False)
print(f"\\nToplam {len(results_df)} satir ({len(THR_MODES)} threshold modu) "
      "-> panel_specific_results.csv")
results_df.head(12)


\n========== Panel: CFTR ==========
--- S1: panel + dengelenmis MASTER cekirdek (625/625) ---
  [S1|CFTR|lightgbm ] test F1(50/50)=0.861  F1(80/20)=0.806+-0.233  P=1.000 R=0.756 thr=0.55
  [S1|CFTR|catboost ] test F1(50/50)=0.875  F1(80/20)=0.850+-0.169  P=1.000 R=0.778 thr=0.50
  [S1|CFTR|nn       ] test F1(50/50)=0.685  F1(80/20)=0.453+-0.162  P=0.893 R=0.556 thr=0.50
  [S1|CFTR|dnn      ] test F1(50/50)=0.618  F1(80/20)=0.422+-0.221  P=0.913 R=0.467 thr=0.51
--- S2: panel + ayni boyutta MASTER (panel-train 2x) ---
  [S2|CFTR|lightgbm ] test F1(50/50)=0.899  F1(80/20)=0.542+-0.099  P=0.909 R=0.889 thr=0.38
  [S2|CFTR|catboost ] test F1(50/50)=0.864  F1(80/20)=0.477+-0.093  P=0.884 R=0.844 thr=0.42
  [S2|CFTR|nn       ] test F1(50/50)=0.871  F1(80/20)=0.565+-0.146  P=0.925 R=0.822 thr=0.51
  [S2|CFTR|dnn      ] test F1(50/50)=0.884  F1(80/20)=0.598+-0.100  P=0.927 R=0.844 thr=0.51
--- S3: NN pretrain(MASTER core) -> finetune(panel) ---
  [S3|CFTR|nn       ] test F1(50/50)=0.747  F1(80

,scenario,panel,model,thr_mode,threshold,best_combo,train_f1,train_precision,train_recall,train_mcc,test_f1_5050,test_precision_5050,test_recall_5050,test_mcc_5050,test_f1_8020_mean,test_f1_8020_std,test_f1_8020_lo,test_f1_8020_hi,n_test
0,S1,CFTR,lightgbm,f1_raw,0.52,,0.964635,0.972686,0.956716,0.928082,0.888889,1.000000,0.800000,0.663325,0.860000,0.190788,0.500000,1.000000,56
1,S1,CFTR,lightgbm,f1_8020,0.55,,0.964313,0.981453,0.947761,0.928562,0.860759,1.000000,0.755556,0.614636,0.806000,0.232732,0.112500,1.000000,56
2,S1,CFTR,lightgbm,mcc_8020,0.55,,0.964313,0.981453,0.947761,0.928562,0.860759,1.000000,0.755556,0.614636,0.806000,0.232732,0.112500,1.000000,56
3,S1,CFTR,catboost,f1_raw,0.40,,0.932749,0.914040,0.952239,0.859534,0.915663,1.000000,0.844444,0.718366,0.894000,0.158000,0.500000,1.000000,56
4,S1,CFTR,catboost,f1_8020,0.50,,0.931138,0.933934,0.928358,0.858942,0.875000,1.000000,0.777778,0.638285,0.850000,0.168819,0.500000,1.000000,56
5,S1,CFTR,catboost,mcc_8020,0.50,,0.931138,0.933934,0.928358,0.858942,0.875000,1.000000,0.777778,0.638285,0.850000,0.168819,0.500000,1.000000,56
6,S1,CFTR,nn,f1_raw,0.39,"{'hidden': 64, 'dropout': 0.3, 'weight_decay':...",0.865837,0.806701,0.934328,0.710703,0.843373,0.921053,0.777778,0.429645,0.546667,0.130494,0.285714,0.666667,56
7,S1,CFTR,nn,f1_8020,0.50,"{'hidden': 64, 'dropout': 0.3, 'weight_decay':...",0.841270,0.898305,0.791045,0.699522,0.684932,0.892857,0.555556,0.224733,0.453333,0.161972,0.064286,0.666667,56
8,S1,CFTR,nn,mcc_8020,0.50,"{'hidden': 64, 'dropout': 0.3, 'weight_decay':...",0.841270,0.898305,0.791045,0.699522,0.684932,0.892857,0.555556,0.224733,0.453333,0.161972,0.064286,0.666667,56
9,S1,CFTR,dnn,f1_raw,0.39,"{'hidden': 128, 'n_layers': 3, 'dropout': 0.4,...",0.836237,0.784314,0.895522,0.645112,0.753247,0.906250,0.644444,0.298425,0.447619,0.190833,0.000000,0.666667,56


In [10]:
# Cell 9: Sonuc Derleme
# Birincil bakis: bootstrap %80/20 pathogenic-F1 (final dagilimi taklit eder).
# Ayrica %50/50 ile kiyas ve threshold-modu karsilastirmasi.

PRIMARY = "test_f1_8020_mean"   # final-realistic birincil metrik

print("=== %80/20 bootstrap pathogenic-F1 (f1_8020 threshold) — scenario x panel x model ===")
sub = results_df[results_df["thr_mode"] == "f1_8020"]
pivot = sub.pivot_table(index=["scenario", "model"], columns="panel",
                        values=PRIMARY, aggfunc="first")
print(pivot.round(4).to_string())

print("\\n=== Her panel icin EN IYI (f1_8020 thr, %80/20 bootstrap F1) ===")
for panel in PANELS:
    s = sub[sub["panel"] == panel].sort_values(PRIMARY, ascending=False).iloc[0]
    print(f"  {panel}: {s['scenario']}/{s['model']}  "
          f"F1(80/20)={s[PRIMARY]:.4f} [{s['test_f1_8020_lo']:.3f}-{s['test_f1_8020_hi']:.3f}]  "
          f"F1(50/50)={s['test_f1_5050']:.3f}  P={s['test_precision_5050']:.3f} "
          f"R={s['test_recall_5050']:.3f}")

print("\\n=== Threshold modu karsilastirmasi (panel-ortalama %80/20 F1) ===")
print(results_df.groupby("thr_mode")[PRIMARY].mean().round(4).to_string())
print("  Yorum: f1_8020/mcc_8020 vs f1_raw -> benign-aware esik final dagilimda "
      "fayda saglıyor mu?")

print("\\n=== Senaryo ortalamasi (f1_8020 thr) ===")
print(sub.groupby("scenario")[[PRIMARY, "test_f1_5050"]].mean().round(4).to_string())

print("\\n=== %50/50 -> %80/20 F1 dususu (dagilim kaymasi etkisi, f1_8020 thr) ===")
sub2 = sub.copy()
sub2["drop_5050_to_8020"] = sub2["test_f1_5050"] - sub2[PRIMARY]
print(sub2.groupby("scenario")["drop_5050_to_8020"].mean().round(4).to_string())


=== %80/20 bootstrap pathogenic-F1 (f1_8020 threshold) — scenario x panel x model ===
panel                CFTR  KANSER     PAH
scenario model                           
S1       catboost  0.8500  0.6820  0.4895
         dnn       0.4224  0.5980  0.4167
         lightgbm  0.8060  0.6507  0.4528
         nn        0.4533  0.5602  0.4180
S2       catboost  0.4766  0.6290  0.4543
         dnn       0.5981  0.5632  0.3625
         lightgbm  0.5417  0.6001  0.3598
         nn        0.5652  0.5237  0.3767
S3       nn        0.5148  0.6221  0.3855
S4       dnn       0.5264  0.6897  0.4052
\n=== Her panel icin EN IYI (f1_8020 thr, %80/20 bootstrap F1) ===
  CFTR: S1/catboost  F1(80/20)=0.8500 [0.500-1.000]  F1(50/50)=0.875  P=1.000 R=0.778
  KANSER: S4/dnn  F1(80/20)=0.6897 [0.480-0.812]  F1(50/50)=0.798  P=0.958 R=0.684
  PAH: S1/catboost  F1(80/20)=0.4895 [0.335-0.533]  F1(50/50)=0.908  P=0.908 R=0.908
\n=== Threshold modu karsilastirmasi (panel-ortalama %80/20 F1) ===
thr_mode
f1_8020     

In [11]:
# Cell 10: Gorsellestirmeler
# Birincil metrik: bootstrap %80/20 pathogenic-F1 (f1_8020 threshold).
SCN_ORDER = ["S1", "S2", "S3", "S4"]
viz = results_df[results_df["thr_mode"] == "f1_8020"].copy()

# --- FIG 1: %80/20 F1 (CI ile) / Precision(50/50) / Recall(50/50), panel x (scn,model) ---
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
metrics = [("test_f1_8020_mean", "Pathogenic F1 (%80/20 bootstrap)"),
           ("test_precision_5050", "Precision (%50/50 test)"),
           ("test_recall_5050", "Recall (%50/50 test)")]
for ax, (metric, title) in zip(axes, metrics):
    piv = viz.pivot_table(index="panel", columns=["scenario", "model"],
                          values=metric, aggfunc="first").reindex(PANELS)
    piv.plot(kind="bar", ax=ax, width=0.85, legend=(metric == metrics[0][0]))
    ax.set_title(title); ax.set_ylim(0, 1.05); ax.set_ylabel(metric)
    ax.set_xlabel("Panel"); ax.tick_params(axis="x", rotation=0); ax.grid(axis="y", alpha=0.3)
    if metric == metrics[0][0]:
        ax.legend(fontsize=7, ncol=2, title="scenario/model")
plt.suptitle("NB15 — Panel-Bazli Metrikler (birincil: %80/20 pathogenic-F1)", fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig1_test_metrics.png"), dpi=110, bbox_inches="tight")
plt.show()

# --- FIG 2: %50/50 vs %80/20 F1 (dagilim kaymasi etkisi) ---
fig, ax = plt.subplots(figsize=(15, 6))
labels = [f"{r.scenario}|{r.panel}|{r.model}" for r in viz.itertuples()]
x = np.arange(len(labels))
ax.bar(x - 0.2, viz["test_f1_5050"], 0.4, label="Test F1 (%50/50)", color="#9ecae1")
ax.bar(x + 0.2, viz["test_f1_8020_mean"], 0.4,
       yerr=viz["test_f1_8020_std"], capsize=2,
       label="Test F1 (%80/20 bootstrap)", color="#fd8d3c")
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=90, fontsize=6)
ax.set_ylabel("Pathogenic F1"); ax.set_ylim(0, 1.05); ax.legend()
ax.set_title("NB15 — %50/50 vs %80/20 Pathogenic F1 (final dagilim kaymasi)", fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig2_train_vs_test.png"), dpi=110, bbox_inches="tight")
plt.show()

# --- FIG 5: Threshold modu karsilastirmasi (panel-ortalama %80/20 F1) ---
fig, ax = plt.subplots(figsize=(9, 5))
mode_piv = results_df.pivot_table(index="thr_mode", values="test_f1_8020_mean",
                                  aggfunc="mean").reindex(["f1_raw", "f1_8020", "mcc_8020"])
ax.bar(mode_piv.index, mode_piv["test_f1_8020_mean"],
       color=["#999999", "#1f77b4", "#2ca02c"])
for i, v in enumerate(mode_piv["test_f1_8020_mean"]):
    ax.text(i, v + 0.005, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("Ort. %80/20 pathogenic-F1"); ax.set_ylim(0, 1.0)
ax.set_title("NB15 — Threshold Modu: benign-aware (f1/mcc_8020) vs eski (f1_raw)",
             fontweight="bold")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig5_threshold_modes.png"), dpi=110, bbox_inches="tight")
plt.show()


In [12]:
# Cell 11: Confusion Matrix'ler + Feature Importance (panel bazli)
from sklearn.metrics import ConfusionMatrixDisplay

# En iyi (scenario,model) her panel icin -> %80/20 bootstrap F1'e gore (f1_8020 thr).
_viz = results_df[results_df["thr_mode"] == "f1_8020"]
best_per_panel = {}
for panel in PANELS:
    s = _viz[_viz["panel"] == panel].sort_values("test_f1_8020_mean", ascending=False).iloc[0]
    best_per_panel[panel] = (s["scenario"], s["model"])

# --- FIG 3: Confusion matrix (her panel icin en iyi model, %50/50 test, f1_8020 thr) ---
fig, axes = plt.subplots(1, len(PANELS), figsize=(6 * len(PANELS), 5))
if len(PANELS) == 1:
    axes = [axes]
for ax, panel in zip(axes, PANELS):
    scn, mdl = best_per_panel[panel]
    res = DETAIL[(scn, panel, mdl)]
    cm = confusion_matrix(res["y_true"], res["y_pred"], labels=[0, 1])
    ConfusionMatrixDisplay(cm, display_labels=["Benign", "Pathogenic"]).plot(
        ax=ax, colorbar=False, cmap="viridis")
    ax.set_title(f"{panel}\n{scn}/{mdl} (thr={res['thr']:.2f}, "
                 f"F1_5050={res['test']['f1']:.3f})")
plt.suptitle("NB15 — Confusion Matrices (panel basina en iyi, %50/50 test)", fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig3_confusion_matrices.png"), dpi=110, bbox_inches="tight")
plt.show()

# --- FIG 4: Feature importance (her panel icin en iyi AGAC modeli) ---
fig, axes = plt.subplots(1, len(PANELS), figsize=(7 * len(PANELS), 6))
if len(PANELS) == 1:
    axes = [axes]
for ax, panel in zip(axes, PANELS):
    cand = _viz[(_viz["panel"] == panel) &
                (_viz["model"].isin(["lightgbm", "catboost"]))].sort_values(
                    "test_f1_8020_mean", ascending=False)
    chosen = None
    for _, r in cand.iterrows():
        res = DETAIL[(r["scenario"], panel, r["model"])]
        if res.get("importance") is not None:
            chosen = (r["scenario"], r["model"], res); break
    if chosen is None:
        ax.set_visible(False); continue
    scn, mdl, res = chosen
    imp = res["importance"].sort_values(ascending=False).head(15)[::-1]
    colors = ["#9467bd" if str(i).startswith("is_missing_") else "#1f77b4" for i in imp.index]
    ax.barh(range(len(imp)), imp.values, color=colors)
    ax.set_yticks(range(len(imp))); ax.set_yticklabels(imp.index, fontsize=8)
    ax.set_title(f"{panel} — {scn}/{mdl}\n(mor = is_missing flag)")
    ax.set_xlabel("Importance")
plt.suptitle("NB15 — Top-15 Feature Importance (panel basina en iyi agac modeli)",
             fontweight="bold")
plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "fig4_feature_importance.png"), dpi=110, bbox_inches="tight")
plt.show()


In [14]:
# Cell 12: PDF Rapor
from fpdf import FPDF
from PIL import Image

class NB15Report(FPDF):
    def header(self):
        self.set_font("Helvetica", "", 9); self.set_text_color(90, 90, 90)
        self.cell(0, 8, "NB15 - Panel-Bazli Egitim & Stacking | TEKNOFEST Genetik Varyant",
                  align="C", ln=True)
        self.set_text_color(0, 0, 0)
    def footer(self):
        self.set_y(-15); self.set_font("Helvetica", "", 8); self.set_text_color(120, 120, 120)
        self.cell(0, 10, f"Sayfa {self.page_no()}", align="C")
    def section(self, title):
        self.ln(2); self.set_fill_color(31, 119, 180); self.set_text_color(255, 255, 255)
        self.set_font("Helvetica", "B", 12); self.cell(0, 9, f"  {title}", ln=True, fill=True)
        self.set_text_color(0, 0, 0); self.ln(2)
    def body(self, text):
        self.set_font("Helvetica", "", 10); self.multi_cell(0, 5.5, text); self.ln(1)
    def table(self, headers, data, widths):
        self.set_font("Helvetica", "B", 8); self.set_fill_color(70, 130, 180)
        self.set_text_color(255, 255, 255)
        for h, w in zip(headers, widths):
            self.cell(w, 7, str(h), border=1, align="C", fill=True)
        self.ln(); self.set_text_color(0, 0, 0); self.set_font("Helvetica", "", 8)
        fill = False
        for row in data:
            self.set_fill_color(235, 240, 248)
            for v, w in zip(row, widths):
                self.cell(w, 6, str(v), border=1, align="C", fill=fill)
            self.ln(); fill = not fill
    def usable_width(self):
        return self.w - self.l_margin - self.r_margin
    def fit_image(self, path, max_h=None):
        iw, ih = Image.open(path).size
        w = self.usable_width(); h = w * ih / iw
        if max_h is not None and h > max_h:
            h = max_h; w = h * iw / ih
        self.image(path, w=w, h=h)

pdf = NB15Report(); pdf.set_auto_page_break(auto=True, margin=18)
viz = results_df[results_df["thr_mode"] == "f1_8020"].copy()

# --- Sayfa 1: Ozet ---
pdf.add_page()
pdf.set_font("Helvetica", "B", 18); pdf.ln(6)
pdf.cell(0, 12, "NB15: Panel-Bazli Egitim & Stacking (v2)", align="C", ln=True)
pdf.set_font("Helvetica", "", 12)
pdf.cell(0, 8, "TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant", align="C", ln=True)
pdf.cell(0, 8, f"Tarih: {datetime.now():%Y-%m-%d %H:%M}", align="C", ln=True); pdf.ln(4)
pdf.section("Deney Tasarimi")
pdf.body(
    "Her panel (CFTR/KANSER/PAH) icin ayri model. Dort senaryo x dort model tipi "
    "(LightGBM, CatBoost, NN, DNN). Birincil metrik: pathogenic (Label=1) F1.\n\n"
    "S1: panel %50 (patho %50 + benign %50) + dengelenmis MASTER cekirdegi "
    f"({MASTER_CORE_POS}/{MASTER_CORE_NEG}).  S2: panel %50 + ayni boyutta MASTER (2x).\n"
    "S3: NN pretrain(MASTER)->finetune(panel).  S4: DNN ile ayni.\n"
    "Test'te MASTER YOK. Imputer/encoder/scaler yalniz train'de fit."
)
pdf.section("KRITIK: Final Test Dagilimi (%80 benign / %20 patho)")
pdf.body(
    "Yarisma final test seti ~%80 benign / %20 patho (train'in TERSI). Bu yuzden:\n"
    "(1) Threshold UC modda secildi: f1_raw (eski, train dengesinde), f1_8020 ve "
    "mcc_8020 (final %80/20 dagilima yeniden orneklenmis train havuzunda).\n"
    "(2) Test iki dagilimda raporlandi: %50/50 (mevcut panel) ve bootstrap %80/20 "
    f"(benign sabit + patho downsample, N={N_BOOT} tekrar, %95 CI). Birincil sonuc: "
    "%80/20 bootstrap pathogenic-F1."
)
pdf.section("NN/DNN Iyilestirme + HPO")
pdf.body(
    f"Agac (LightGBM/CatBoost): {N_CV}-fold CV grid search (LGBM/CB 12 kombo). "
    f"NN/DNN: {NN_CV}-fold CV + FocalLoss (gamma=2) + class weighting + validasyon-bazli "
    "early stopping + kucuk/duzenli mimari (SmallMLP, yuksek dropout, weight_decay). "
    "Bu degisiklikler NB15-v1'deki NN/DNN overfit'ine (train-test farki ~0.15) yanit."
)
pdf.section("Missing & Veri Butunlugu")
dup_txt = ", ".join(f"{p}={dup_report[p]}" for p in PANELS)
pdf.body(
    f"M3 strateji (NB14): >%{int(HIGH_MISSING_THRESHOLD*100)} NaN sutunlar icin is_missing "
    "flag + tum sayisal sutunlar medyan impute. "
    f"Cross-panel birebir-ayni satir drop: {dup_txt}. "
    f"{len(drop_cols)} sutun (sabit+ozdes) drop -> {len(feature_cols)} feature (FE yok)."
)

# --- Sayfa 2: Ana sonuc tablosu (f1_8020, %50/50 vs %80/20) ---
pdf.add_page()
pdf.section("1. Sonuclar (f1_8020 threshold) - %50/50 vs %80/20 bootstrap")
headers = ["Scn", "Panel", "Model", "Thr", "Te-F1(50/50)", "Te-P", "Te-R",
           "F1(80/20)", "80/20 CI"]
widths = [13, 18, 18, 12, 26, 16, 16, 22, 30]
data = []
for r in viz.itertuples():
    data.append([r.scenario, r.panel, r.model, f"{r.threshold:.2f}",
                 f"{r.test_f1_5050:.3f}", f"{r.test_precision_5050:.3f}",
                 f"{r.test_recall_5050:.3f}", f"{r.test_f1_8020_mean:.3f}",
                 f"[{r.test_f1_8020_lo:.2f}-{r.test_f1_8020_hi:.2f}]"])
pdf.table(headers, data, widths)

# --- Sayfa 2b: TRAIN sonuclari (overfit kontrolu) ---
pdf.ln(3)
pdf.section("1b. Train Sonuclari (f1_8020 thr) + Train-Test F1 farki")
th = ["Scn", "Panel", "Model", "Tr-F1", "Tr-P", "Tr-R", "Tr-MCC",
      "Te-F1(50/50)", "Tr-Te farki"]
tw = [13, 18, 18, 18, 16, 16, 18, 26, 24]
td = []
for r in viz.itertuples():
    gap = r.train_f1 - r.test_f1_5050
    td.append([r.scenario, r.panel, r.model, f"{r.train_f1:.3f}",
               f"{r.train_precision:.3f}", f"{r.train_recall:.3f}", f"{r.train_mcc:.3f}",
               f"{r.test_f1_5050:.3f}", f"{gap:+.3f}"])
pdf.table(th, td, tw)
pdf.ln(2)
_mean_gap = (viz["train_f1"] - viz["test_f1_5050"]).mean()
pdf.body(f"Ortalama train-test F1 farki (overfit gostergesi): {_mean_gap:.3f}. "
         "Yuksek fark = overfit. NN/DNN iyilestirmesi bu farki dusurmeyi hedefler.")

# --- Sayfa 3: Threshold modu + panel en iyi + NN combo ---
pdf.add_page()
pdf.section("2. Threshold Modu Karsilastirmasi (panel-ort. %80/20 F1)")
mode_means = results_df.groupby("thr_mode")["test_f1_8020_mean"].mean()
pdf.table(["Threshold modu", "Ort. %80/20 F1"],
          [[m, f"{mode_means.get(m, float('nan')):.4f}"] for m in ["f1_raw", "f1_8020", "mcc_8020"]],
          [60, 60])
pdf.ln(2)
pdf.body("f1_raw = eski (train dengesinde F1-max). f1_8020 / mcc_8020 = benign-aware "
         "(final %80/20 dagilimda esik secimi). Yuksek olan, final test icin tercih edilmeli.")
pdf.section("3. Her Panel Icin En Iyi (%80/20 bootstrap F1, f1_8020 thr)")
bp = []
for panel in PANELS:
    s = viz[viz["panel"] == panel].sort_values("test_f1_8020_mean", ascending=False).iloc[0]
    bp.append([panel, f"{s['scenario']}/{s['model']}", f"{s['test_f1_8020_mean']:.4f}",
               f"[{s['test_f1_8020_lo']:.2f}-{s['test_f1_8020_hi']:.2f}]",
               f"{s['test_f1_5050']:.3f}"])
pdf.table(["Panel", "En iyi", "F1(80/20)", "80/20 CI", "F1(50/50)"], bp, [28, 42, 32, 40, 32])
pdf.ln(2)
pdf.section("3b. Secilen NN/DNN Hiperparametreleri (CV grid search)")
nn_rows = [[r.scenario, r.panel, r.model, r.best_combo]
           for r in viz[viz["model"].isin(["nn", "dnn"])].itertuples()]
if nn_rows:
    pdf.table(["Scn", "Panel", "Model", "Secilen combo"], nn_rows, [16, 22, 16, 130])

# --- Grafikler: genis figurler landscape + clamp ---
WIDE = {"fig1_test_metrics.png", "fig3_confusion_matrices.png", "fig4_feature_importance.png"}
for fig_name, title in [
    ("fig1_test_metrics.png", "4. Metrikler (%80/20 F1, Precision, Recall)"),
    ("fig2_train_vs_test.png", "5. %50/50 vs %80/20 F1 (dagilim kaymasi)"),
    ("fig5_threshold_modes.png", "6. Threshold Modu Karsilastirmasi"),
    ("fig3_confusion_matrices.png", "7. Confusion Matrix'ler (panel basina en iyi)"),
    ("fig4_feature_importance.png", "8. Feature Importance (panel basina en iyi agac)"),
]:
    path = os.path.join(RESULTS_DIR, fig_name)
    if os.path.exists(path):
        orient = "L" if fig_name in WIDE else "P"
        pdf.add_page(orientation=orient)
        pdf.section(title)
        pdf.fit_image(path, max_h=pdf.h - pdf.get_y() - 20)

out_pdf = os.path.join(REPORTS_DIR, "NB15_panel_specific_report.pdf")
pdf.output(out_pdf)
print(f"PDF rapor yazildi: {out_pdf}")
print(f"CSV: {os.path.join(RESULTS_DIR, 'panel_specific_results.csv')}")


PDF rapor yazildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\NB15_panel_specific_report.pdf
CSV: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v5_panel_specific\panel_specific_results.csv
